In [ ]:
import numpy as np
import re
from pathlib import Path
from PIL import Image
from scipy.optimize import curve_fit

# ===============================
# USER SETTINGS
# ===============================
PIXEL_SIZE = 11e-6        # Kamera-Pitch [m]  (oder effektiver Pixel in der Strahlebene!)
LAMBDA = 800e-9          # Wellenlänge [m]
THRESHOLD = 20            # Kamera-Offset

# ===============================
# Load TIFF
# ===============================
def load_img(path):
    I = np.array(Image.open(path), dtype=np.float64)
    I[I < THRESHOLD] = 0.0
    return I

# ===============================
# ISO second moments
# ===============================
def d4sigma_xy(I, dx):
    h, w = I.shape
    x = (np.arange(w) - w/2) * dx
    y = (np.arange(h) - h/2) * dx
    X, Y = np.meshgrid(x, y)

    P = I / I.sum()
    mx = np.sum(X * P)
    my = np.sum(Y * P)

    sx = np.sqrt(np.sum(P * (X - mx)**2))
    sy = np.sqrt(np.sum(P * (Y - my)**2))

    return 4*sx, 4*sy

# ===============================
# Filename → z [m]
# ===============================
def get_z(filename):
    m = re.search(r"(\d+)\s*cm", filename)
    return float(m.group(1)) / 100.0

# ===============================
# Fit model
# ===============================
def beam_model(z, w0, z0, M2):
    return w0**2 + ((M2 * LAMBDA)/(np.pi*w0))**2 * (z - z0)**2

# ===============================
# MAIN
# ===============================
files = sorted(Path(".").glob("messung*cm.tiff"))

z = []
wx = []
wy = []

print("\nLoaded files:")
for f in files:
    I = load_img(f)
    Dx, Dy = d4sigma_xy(I, PIXEL_SIZE)
    z.append(get_z(f.name))
    wx.append(Dx/2)
    wy.append(Dy/2)
    print(f"{f.name:20s}  z={z[-1]:.2f} m   Dx={Dx*1e3:.3f} mm   Dy={Dy*1e3:.3f} mm")

z = np.array(z)
wx = np.array(wx)
wy = np.array(wy)

# ===============================
# Fit X and Y
# ===============================
p0 = [np.min(wx), z[np.argmin(wx)], 1.5]

px, _ = curve_fit(beam_model, z, wx**2, p0=p0)
py, _ = curve_fit(beam_model, z, wy**2, p0=p0)

w0x, z0x, M2x = px
w0y, z0y, M2y = py

print("\n================== ISO M² ==================")
print(f"M²_x = {M2x:.3f}")
print(f"M²_y = {M2y:.3f}")
print(f"w0_x = {w0x*1e3:.3f} mm   z0_x = {z0x:.3f} m")
print(f"w0_y = {w0y*1e3:.3f} mm   z0_y = {z0y:.3f} m")
print("===========================================\n")



Loaded files:
messung 200 cm.tiff   z=2.00 m   Dx=4.649 mm   Dy=4.870 mm
messung 350cm.tiff    z=3.50 m   Dx=4.807 mm   Dy=5.006 mm
messung 500cm.tiff    z=5.00 m   Dx=4.965 mm   Dy=5.064 mm
messung 53cm.tiff     z=0.53 m   Dx=4.756 mm   Dy=4.633 mm
messung 750cm.tiff    z=7.50 m   Dx=5.086 mm   Dy=5.163 mm

================== ISO M² ==================
M²_x = 1.133
M²_y = 0.000
w0_x = 2.348 mm   z0_x = -0.680 m
w0_y = 0.002 mm   z0_y = -66.652 m

